<a href="https://colab.research.google.com/github/sahbanovmazid2/autoservice-project/blob/main/AutoService_Montage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Автосервис


In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO_URL = 'https://github.com/sahbanovmazid2/autoservice-project.git'
PROJECT = Path('/content/autoservice-project')
if PROJECT.exists():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print('GitHub:', REPO_URL)
print('Каталог:', Path.cwd())
print('Файлы:', ', '.join(sorted(p.name for p in PROJECT.iterdir() if p.is_file())))

In [ ]:
import platform
print('Python:', platform.python_version())
for folder in [PROJECT, PROJECT/'tests', PROJECT/'docs']:
    print('\n'+str(folder.relative_to(PROJECT.parent))+':')
    for p in sorted(folder.iterdir()):
        if p.is_file() and not p.name.startswith('.'):
            print(' •', p.name)

## Структура базы данных


In [ ]:
from IPython.display import display, Image
diagram = PROJECT/'docs'/'er_diagram.png'
assert diagram.is_file(), 'В репозитории отсутствует docs/er_diagram.png'
display(Image(filename=str(diagram), width=980))

In [ ]:
from seed import seed
from database import init_database, list_clients, list_cars, list_services, list_orders
DEMO_DB = Path('/content/autoservice_majid_colab.db')
if DEMO_DB.exists(): DEMO_DB.unlink()  # воспроизводимый запуск
created = seed(DEMO_DB)
print('База:', DEMO_DB)
print('Данные созданы:', created)
print('Клиентов:', len(list_clients(DEMO_DB)))
print('Автомобилей:', len(list_cars(DEMO_DB)))
print('Услуг:', len(list_services(DEMO_DB)))
print('Заказов:', len(list_orders(DEMO_DB)))

In [ ]:
import sqlite3
with sqlite3.connect(DEMO_DB) as conn:
    tables = [x[0] for x in conn.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name")]
    for name in tables:
        columns = [x[1] for x in conn.execute(f'PRAGMA table_info({name})')]
        fk = conn.execute(f'PRAGMA foreign_key_list({name})').fetchall()
        print(f'{name}: {", ".join(columns)} | внешних ключей: {len(fk)}')
assert len(tables) == 5, tables

In [ ]:
import pandas as pd
from IPython.display import display
print('КЛИЕНТЫ:'); display(pd.DataFrame(list_clients(DEMO_DB)))
print('АВТОМОБИЛИ И ВЛАДЕЛЬЦЫ:'); display(pd.DataFrame(list_cars(DEMO_DB)))
print('ЗАКАЗЫ И ИТОГОВЫЕ СУММЫ:'); display(pd.DataFrame(list_orders(DEMO_DB)))

In [ ]:
from database import add_client, update_client, delete_client
client_id = add_client('Петров Максим', '+79009998877', 'petrov@example.com', DEMO_DB)
print('CREATE:', client_id)
print('READ:', next(x for x in list_clients(DEMO_DB) if x['id'] == client_id))
print('UPDATE:', update_client(client_id, 'Петров Максим Сергеевич', '+79009998877', DEMO_DB))
print('READ после изменения:', next(x for x in list_clients(DEMO_DB) if x['id'] == client_id))
print('DELETE:', delete_client(client_id, DEMO_DB))
assert all(x['id'] != client_id for x in list_clients(DEMO_DB))
print('CRUD успешно выполнен')

In [ ]:
from database import create_order, add_service_to_order, order_details, set_order_status
cars = list_cars(DEMO_DB); services = list_services(DEMO_DB)
car_id = cars[0]['id']; service_id = services[0]['id']
order_id = create_order(car_id, 'Дополнительное техническое обслуживание', DEMO_DB)
add_service_to_order(order_id, service_id, quantity=2, db_path=DEMO_DB)
print('Новый заказ:', order_id)
display(pd.DataFrame(order_details(order_id, DEMO_DB)))
print('Стоимость:', next(x['total'] for x in list_orders(DEMO_DB) if x['id'] == order_id))
print('Смена статуса:', set_order_status(order_id, 'в работе', DEMO_DB))

In [ ]:
from database import search_cars, popular_services, active_clients, service_stats
print('Поиск Toyota:'); display(pd.DataFrame(search_cars('Toyota', DEMO_DB)))
print('Популярные услуги:'); display(pd.DataFrame(popular_services(DEMO_DB)))
print('Активные клиенты:'); display(pd.DataFrame(active_clients(DEMO_DB)))
print('Статистика:', service_stats(DEMO_DB))

In [ ]:
with sqlite3.connect(DEMO_DB) as conn:
    query = """SELECT s.name AS service, SUM(os.quantity) AS uses,
           ROUND(SUM(os.quantity * os.unit_price), 2) AS revenue
    FROM services s
    JOIN order_services os ON os.service_id = s.id
    GROUP BY s.id
    HAVING SUM(os.quantity) >= 1
    ORDER BY revenue DESC"""
    display(pd.read_sql_query(query, conn))

In [ ]:
from reports import export_orders_csv, export_services_txt, backup_database
CSV_PATH = Path('/content/majid_orders.csv')
TXT_PATH = Path('/content/majid_services_report.txt')
BACKUP_PATH = Path('/content/majid_autoservice_backup.db')
print('CSV:', export_orders_csv(CSV_PATH, DEMO_DB))
print('TXT:', export_services_txt(TXT_PATH, DEMO_DB))
print('Backup:', backup_database(BACKUP_PATH, DEMO_DB))
print('Размеры:', {p.name: p.stat().st_size for p in (CSV_PATH, TXT_PATH, BACKUP_PATH)})
print('\nНачало TXT-отчёта:\n', TXT_PATH.read_text(encoding='utf-8')[:650])

In [ ]:
with sqlite3.connect(DEMO_DB) as conn:
    print('Основная БД:', conn.execute('PRAGMA integrity_check').fetchone()[0])
    source_count = conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0]
with sqlite3.connect(BACKUP_PATH) as conn:
    print('Резервная БД:', conn.execute('PRAGMA integrity_check').fetchone()[0])
    backup_count = conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0]
print('Заказов в оригинале / копии:', source_count, backup_count)
assert source_count == backup_count

In [ ]:
from validators import validate_year, validate_quantity, validate_price, validate_order_status
cases = [(validate_quantity, 0), (validate_quantity, -1), (validate_price, -100),
         (validate_year, 1900), (validate_order_status, 'отменён')]
for fn, value in cases:
    try:
        fn(value)
        print('ОШИБКА: принято недопустимое значение', fn.__name__, value)
    except (ValueError, TypeError) as exc:
        print('Ожидаемый отказ:', fn.__name__, repr(value), '→', type(exc).__name__, str(exc))

In [ ]:
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', '*unittest.py', '-v'], cwd=PROJECT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
assert result.returncode == 0, 'unittest завершился с ошибкой'

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', str(PROJECT/'requirements.txt')], check=True)
import pytest
print('pytest:', pytest.__version__)

In [ ]:
result = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'], cwd=PROJECT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
assert result.returncode == 0, 'pytest завершился с ошибкой'

In [ ]:
result = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '--cov=database', '--cov=validators', '--cov=reports', '--cov-report=term-missing'], cwd=PROJECT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
assert result.returncode == 0, 'Проверка покрытия завершилась с ошибкой'

In [ ]:
import matplotlib.pyplot as plt
orders = pd.DataFrame(list_orders(DEMO_DB))
counts = orders['status'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8,4))
counts.plot(kind='bar', ax=ax)
ax.set_title('Заказы автосервиса по статусам')
ax.set_xlabel('Статус заказа'); ax.set_ylabel('Количество')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
required = ['database.py','validators.py','seed.py','reports.py','app.py','schema.dbml','docs/er_diagram.png','tests/test_database_unittest.py','tests/test_validators_pytest.py','tests/test_workflow_pytest.py']
missing = [f for f in required if not (PROJECT/f).exists()]
print('Отсутствующие файлы:', missing)
print('Всего таблиц:', len(tables), '| заказов:', len(list_orders(DEMO_DB)))
print('Файлы экспорта готовы:', all(p.exists() for p in (CSV_PATH, TXT_PATH, BACKUP_PATH)))
assert not missing
print('ИТОГ: проект AutoService Manager выполнен успешно')